In [1]:
import pandas as pd
train = pd.read_csv("D:\\File\\train_dataset\\train_set.csv")
test = pd.read_csv("D:\\File\\test_dataset\\test_set.csv")


In [2]:
print(train.shape)

(842424, 4)


In [9]:
journal_dataset = pd.read_csv("D:\\File\\train_dataset\\journal_full_info.csv")
journal_dataset.head()

,Journal,Aims,Label,Rank by SJR,Best Rank,Title_modified,URL,URL_Scimago,SJR index,Best Quartile,...,Best Categories,"Biochemistry, Genetics and Molecular Biology",Medicine,Immunology and Microbiology,"Pharmacology, Toxicology and Pharmaceutics",Neuroscience,Health Professions,Nursing,Psychology,Dentistry
0,Therapeutic Advances in Neurological Disorders,Therapeutic Advances in Neurological Disorders...,1340,1041,695,Ther+Adv+Neurol+Disord,https://pubmed.ncbi.nlm.nih.gov/?term=%22Ther+...,https://www.scimagojr.com/journalsearch.php?q=...,"1,436",Q1,...,['Neurology (clinical)']; ['Neurology']; ['Pha...,0,1,0,1,1,0,0,0,0
1,International Journal of Infectious Diseases,The International Journal of Infectious Diseas...,694,1042,696,Int+J+Infect+Dis,https://pubmed.ncbi.nlm.nih.gov/?term=%22Int+J...,https://www.scimagojr.com/journalsearch.php?q=...,"1,435",Q1,...,"['Infectious Diseases', 'Medicine (miscellaneo...",0,1,0,0,0,0,0,0,0
2,Acta Physiologica,Acta Physiologica is an important forum for th...,22,1043,336,Acta+Physiol+Acad+Sci+Hung,https://pubmed.ncbi.nlm.nih.gov/?term=%22Acta+...,https://www.scimagojr.com/journalsearch.php?q=...,"1,433",Q1,...,['Physiology'],1,0,0,0,0,0,0,0,0
3,Psychology and Aging,Psychology and Aging publishes original artic...,1224,1044,697,Psychol+Aging,https://pubmed.ncbi.nlm.nih.gov/?term=%22Psych...,https://www.scimagojr.com/journalsearch.php?q=...,"1,433",Q2,...,['Aging']; ['Geriatrics and Gerontology']; ['S...,1,1,0,0,0,0,0,1,0
4,Endocrinology and Metabolism Clinics of North ...,Endocrinology and Metabolism Clinics of North ...,484,1045,699,Endocrinol+Metab+Clin+North+Am,https://pubmed.ncbi.nlm.nih.gov/?term=%22Endoc...,https://www.scimagojr.com/journalsearch.php?q=...,"1,432",Q1,...,"['Endocrinology']; ['Endocrinology, Diabetes a...",1,1,0,0,0,0,0,0,0


In [3]:
samples = train.sample(100)
samples

,Title,Abstract,Keywords,Label
431140,Human microglia phenotypes in the brain associ...,Cognitive impairment in individuals infected w...,NaN,411
823330,The Effects of Whole Corneal and Whole Eye Hig...,Purpose: To evaluate the effect of different w...,NaN,912
196649,Genomic analyses of gray fox lineages suggest ...,The gray fox Urocyon cinereoargenteus lineage ...,Urocyon cinereoargenteus; genomics; gray fox; ...,839
142135,A perfect fit: Bacteriophage receptor binding ...,Bacteriophages are the most abundant biologica...,NaN,410
469265,Structures of topoisomerase V in complex with ...,Topoisomerase V is a unique topoisomerase that...,Methanopyrus kandleri; archaea; methanogen; mo...,1393
...,...,...,...,...
290903,High Endothelial Venules Promote Permissive T ...,Tumoral high endothelial venule TU HEV formati...,NaN,254
98430,Stimulation of the ventromedial prefrontal cor...,The ventromedial prefrontal cortex vm PFC medi...,NaN,1355
229295,Humoral and Cellular Responses to COVID 19 Vac...,Despite the high efficacy of the BNT162b2 vacc...,SARS Co V 2; antibodies; cellular immunity; nu...,1378
806659,Bifunctional Oxygen Electrocatalysis on Mixed ...,Non precious metal catalysts are promising alt...,anion exchange membrane fuel cell; carbon nano...,5


In [2]:
import torch
print(torch.__version__)

2.10.0+cpu


In [4]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import pandas as pd


df = pd.DataFrame(samples)

# ===== Tokenize và tạo embeddings =====
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def create_token_sequence(row, tokenizer, model, max_length=512):
    """
    Ghép Title, Abstract, Keywords thành một chuỗi,
    tokenize rồi lấy last_hidden_state (embeddings).
    
    Return:
      tokens: [seq_len, hidden_dim]
      token_ids: [seq_len]
      segment_info: {'title': (start, end), 'abstract': (...), ...}
    """
    segments = []
    texts = []
    segment_info = {}
    
    # Title
    title = str(row['Title']) if pd.notna(row['Title']) else ""
    texts.append(title)
    segment_info['title'] = (len(segments), len(segments) + len(title.split()))
    segments.append(title)
    
    # Abstract
    abstract = str(row['Abstract']) if pd.notna(row['Abstract']) else ""
    texts.append(abstract)
    segment_info['abstract'] = (len(segments), len(segments) + len(abstract.split()))
    segments.append(abstract)
    
    # Keywords
    keywords = str(row['Keywords']) if pd.notna(row['Keywords']) and row['Keywords'] != 'NaN' else ""
    texts.append(keywords)
    segment_info['keywords'] = (len(segments), len(segments) + len(keywords.split()))
    segments.append(keywords)
    
    # Ghép tất cả
    full_text = ' [SEP] '.join(segments)
    
    # Tokenize
    encoded = tokenizer(
        full_text,
        max_length=max_length,
        truncation=True,
        padding='max_length',
        return_tensors='pt'
    )
    
    # Lấy embeddings
    with torch.no_grad():
        outputs = model(**encoded)
    
    embeddings = outputs.last_hidden_state[0]  # [seq_len, hidden_dim]
    token_ids = encoded['input_ids'][0]
    
    return embeddings, token_ids, segment_info


def token_merging_weighted(embeddings, merge_ratio=0.3, keep_special=True):
    """
    Token Merging với trọng số (ưu tiên giữ token tìm được cao trong Abstract).
    
    embeddings: [N, D]
    merge_ratio: tỷ lệ token cần merge (0.3 = merge 30%)
    keep_special: giữ [CLS], [SEP], [PAD]
    
    Return:
      merged: [N', D] (N' < N)
      mask_merged: [N] boolean (True = bị merge)
    """
    N, D = embeddings.shape
    device = embeddings.device
    
    # Xác định token cần loại (padding, special token)
    if keep_special:
        protected = torch.zeros(N, dtype=torch.bool, device=device)
        # Token 0 (CLS), chỉ số có giá trị nhỏ (PAD token) thường là 0-103
        # Ở đây đơn giản: giữ token 0 (CLS)
        protected[0] = True
        # Tìm [SEP] token (thường id 102 trong BERT)
        sep_positions = torch.where(embeddings.abs().sum(dim=1) < 0.01)[0]  # heuristic for padding
    else:
        protected = torch.zeros(N, dtype=torch.bool, device=device)
    
    # Normalize để tính cosine similarity
    z = F.normalize(embeddings, p=2, dim=-1)  # [N, D]
    sim = torch.mm(z, z.t())                   # [N, N]
    
    # Mask để không so sánh token với chính nó và các protected token
    mask = torch.eye(N, device=device).bool()
    mask = mask | protected.unsqueeze(0) | protected.unsqueeze(1)
    sim = sim.masked_fill(mask, -1e9)
    
    # Lấy các cặp i<j với similarity cao nhất
    num_pairs_to_merge = max(1, int(N * merge_ratio // 2))
    
    iu, ju = torch.triu_indices(N, N, offset=1, device=device)
    pair_sim = sim[iu, ju]  # [num_pairs]
    
    _, idx = torch.topk(pair_sim, k=min(num_pairs_to_merge, len(pair_sim)), largest=True)
    
    merged_embeddings = []
    mask_merged = torch.zeros(N, dtype=torch.bool, device=device)
    used = torch.zeros(N, dtype=torch.bool, device=device)
    
    # Ghép các cặp tương tự
    for p_idx in idx.tolist():
        i = iu[p_idx].item()
        j = ju[p_idx].item()
        
        if used[i] or used[j] or protected[i] or protected[j]:
            continue
        
        # Merge i vào j (trung bình cosine)
        merged = (embeddings[i] + embeddings[j]) / 2
        merged_embeddings.append((j, merged))
        mask_merged[i] = True
        used[i] = True
    
    # Tạo kết quả
    result = embeddings.clone()
    for j, merged in merged_embeddings:
        result[j] = merged
    
    # Loại bỏ các token đã merge
    result = result[~mask_merged]
    
    return result, mask_merged


# ===== Demo =====
print("=" * 70)
print("TOKEN MERGING TRÊN TẬP SAMPLES")
print("=" * 70)

for idx, row in df.iterrows():
    print(f"\n🔹 Sample {idx + 1}: Title='{row['Title'][:50]}...'")
    print(f"   Label: {row['Label']}")
    
    # Tokenize
    embeddings, token_ids, seg_info = create_token_sequence(row, tokenizer, model)
    print(f"   Original token count: {embeddings.shape[0]}")
    
    # Merge 30% token
    merged_emb, mask = token_merging_weighted(embeddings, merge_ratio=0.3, keep_special=True)
    print(f"   After merging (30%):  {merged_emb.shape[0]}")
    print(f"   Reduction: {(1 - merged_emb.shape[0] / embeddings.shape[0]) * 100:.1f}%")
    
    # Đã có embeddings đã merge, có thể feed vào classifier
    final_embedding = merged_emb.mean(dim=0)  # pooling toàn bộ
    print(f"   Final pooled embedding: {final_embedding.shape[0]} dims")
    print()

print("=" * 70)

d:\File\KLTN\kl_test\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\File\KLTN\kl_test\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ACER\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see th

TOKEN MERGING TRÊN TẬP SAMPLES

🔹 Sample 431141: Title='Human microglia phenotypes in the brain associated...'
   Label: 411
   Original token count: 512
   After merging (30%):  458
   Reduction: 10.5%
   Final pooled embedding: 768 dims


🔹 Sample 823331: Title='The Effects of Whole Corneal and Whole Eye Higher ...'
   Label: 912
   Original token count: 512
   After merging (30%):  470
   Reduction: 8.2%
   Final pooled embedding: 768 dims


🔹 Sample 196650: Title='Genomic analyses of gray fox lineages suggest anci...'
   Label: 839
   Original token count: 512
   After merging (30%):  478
   Reduction: 6.6%
   Final pooled embedding: 768 dims


🔹 Sample 142136: Title='A perfect fit: Bacteriophage receptor binding prot...'
   Label: 410
   Original token count: 512
   After merging (30%):  463
   Reduction: 9.6%
   Final pooled embedding: 768 dims


🔹 Sample 469266: Title='Structures of topoisomerase V in complex with DNA ...'
   Label: 1393
   Original token count: 512
   After mer